<p align="right"><a href="./C1_W3_Lab06_Gradient_Descent_Soln.ipynb">English</a> | <strong>中文</strong></p>

# Optional Lab: 逻辑回归的梯度下降（Gradient Descent）

## 目标
在本实验中，你将：
- 为逻辑回归更新梯度下降。
- 在一个熟悉的数据集上探索梯度下降

In [1]:
import copy, math
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from lab_utils_common import  dlc, plot_data, plt_tumor_data, sigmoid, compute_cost_logistic
from plt_quad_logistic import plt_quad_logistic, plt_prob
plt.style.use('./deeplearning.mplstyle')

## 数据集
我们从决策边界实验用过的同一个双特征数据集开始。

In [2]:
X_train = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_train = np.array([0, 0, 0, 1, 1, 1])

和之前一样，我们用一个辅助函数把数据画出来。标签 $y=1$ 的数据点显示为红色叉号，标签 $y=0$ 的数据点显示为蓝色圆圈。

In [3]:
fig,ax = plt.subplots(1,1,figsize=(4,4))
plot_data(X_train, y_train, ax)

ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 逻辑回归的梯度下降
<img align="right" src="./images/C1_W3_Logistic_gradient_descent.png"     style=" width:400px; padding: 10px; " >

回忆一下，梯度下降算法用到梯度的计算：
$$\begin{align*}
&\text{repeat until convergence:} \; \lbrace \\
&  \; \; \;w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{for j := 0..n-1} \\
&  \; \; \;  \; \;b = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b} \\
&\rbrace
\end{align*}$$

其中每次迭代对所有 $j$ 的 $w_j$ 同时更新，且
$$\begin{align*}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{2} \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{3}
\end{align*}$$

* m 是数据集中的训练样本数量
* $f_{\mathbf{w},b}(x^{(i)})$ 是模型的预测值，$y^{(i)}$ 是目标值
* 对逻辑回归模型
    $z = \mathbf{w} \cdot \mathbf{x} + b$
    $f_{\mathbf{w},b}(x) = g(z)$
    其中 $g(z)$ 是 sigmoid 函数：
    $g(z) = \frac{1}{1+e^{-z}}$

# 梯度下降的实现
梯度下降算法的实现有两部分：
- 实现上面式 (1) 的循环。即下面的 `gradient_descent`，在 optional 和 practice 实验中通常已给好。
- 计算当前梯度，即上面的式 (2)、(3)。即下面的 `compute_gradient_logistic`。本周的 practice lab 会要求你实现它。

#### 计算梯度——代码说明
对所有 $w_j$ 和 $b$ 实现上面的式 (2)、(3)。实现方式有很多，下面这样：
- 初始化用来累加 `dj_dw` 和 `dj_db` 的变量
- 对每个样本
    - 计算该样本的误差 $g(\mathbf{w} \cdot \mathbf{x}^{(i)} + b) - \mathbf{y}^{(i)}$
    - 对该样本中的每个输入值 $x_{j}^{(i)}$，
        - 把误差乘以输入 $x_{j}^{(i)}$，加到 `dj_dw` 的对应元素上（上面式 2）
    - 把误差加到 `dj_db` 上（上面式 3）
- 把 `dj_db` 和 `dj_dw` 都除以样本总数 (m)
- 注意 numpy 中 $\mathbf{x}^{(i)}$ 是 `X[i,:]` 或 `X[i]`，$x_{j}^{(i)}$ 是 `X[i,j]`

In [11]:
def compute_gradient_logistic(X, y, w, b): 
    """
    Computes the gradient for logistic regression 
 
    Args:
      X (ndarray (m,n): Data, m examples with n features
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
    Returns
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar)      : The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape
    dj_dw = np.zeros((n,))                           #(n,)
    dj_db = 0.

    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i],w) + b)          #(n,)(n,)=scalar
        err_i  = f_wb_i  - y[i]                       #scalar
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i,j]      #scalar
        dj_db = dj_db + err_i
    dj_dw = dj_dw/m                                   #(n,)
    dj_db = dj_db/m                                   #scalar
        
    return dj_db, dj_dw  

用下面的 cell 检查梯度函数的实现。

In [12]:
X_tmp = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_tmp = np.array([0, 0, 0, 1, 1, 1])
w_tmp = np.array([2.,3.])
b_tmp = 1.
dj_db_tmp, dj_dw_tmp = compute_gradient_logistic(X_tmp, y_tmp, w_tmp, b_tmp)
print(f"dj_db: {dj_db_tmp}" )
print(f"dj_dw: {dj_dw_tmp.tolist()}" )

dj_db: 0.49861806546328574
dj_dw: [0.498333393278696, 0.49883942983996693]


**期望输出**
```
dj_db: 0.49861806546328574
dj_dw: [0.498333393278696, 0.49883942983996693]
```

#### 梯度下降代码
实现上面式 (1) 的代码如下。花点时间找到并对照代码里的函数与上面的方程。

In [13]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters): 
    """
    Performs batch gradient descent
    
    Args:
      X (ndarray (m,n)   : Data, m examples with n features
      y (ndarray (m,))   : target values
      w_in (ndarray (n,)): Initial values of model parameters  
      b_in (scalar)      : Initial values of model parameter
      alpha (float)      : Learning rate
      num_iters (scalar) : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,))   : Updated values of parameters
      b (scalar)         : Updated value of parameter 
    """
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):
        # Calculate the gradient and update the parameters
        dj_db, dj_dw = compute_gradient_logistic(X, y, w, b)   

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            J_history.append( compute_cost_logistic(X, y, w, b) )

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]}   ")
        
    return w, b, J_history         #return final w,b and J history for graphing


我们在数据集上运行梯度下降。

In [14]:
w_tmp  = np.zeros_like(X_train[0])
b_tmp  = 0.
alph = 0.1
iters = 10000

w_out, b_out, _ = gradient_descent(X_train, y_train, w_tmp, b_tmp, alph, iters) 
print(f"\nupdated parameters: w:{w_out}, b:{b_out}")

Iteration    0: Cost 0.684610468560574   
Iteration 1000: Cost 0.1590977666870456   
Iteration 2000: Cost 0.08460064176930081   
Iteration 3000: Cost 0.05705327279402531   
Iteration 4000: Cost 0.042907594216820076   
Iteration 5000: Cost 0.034338477298845684   
Iteration 6000: Cost 0.028603798022120097   
Iteration 7000: Cost 0.024501569608793   
Iteration 8000: Cost 0.02142370332569295   
Iteration 9000: Cost 0.019030137124109114   

updated parameters: w:[5.28 5.08], b:-14.222409982019837


#### 我们把梯度下降的结果画出来：

In [15]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
# plot the probability 
plt_prob(ax, w_out, b_out)

# Plot the original data
ax.set_ylabel(r'$x_1$')
ax.set_xlabel(r'$x_0$')   
ax.axis([0, 4, 0, 3.5])
plot_data(X_train,y_train,ax)

# Plot the decision boundary
x0 = -b_out/w_out[0]
x1 = -b_out/w_out[1]
ax.plot([0,x0],[x1,0], c=dlc["dlblue"], lw=1)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

上图中：
 - 阴影反映 y=1 的概率（决策边界之前的结果）
 - 决策边界是概率 = 0.5 的那条线

## 另一个数据集
我们回到一个单变量数据集。只有两个参数 $w$、$b$ 时，可以用等高线图画出代价函数，以便更好地理解梯度下降在做什么。

In [17]:
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0,  0, 0, 1, 1, 1])

和之前一样，我们用一个辅助函数把数据画出来。标签 $y=1$ 的数据点显示为红色叉号，标签 $y=0$ 的数据点显示为蓝色圆圈。

In [18]:
fig,ax = plt.subplots(1,1,figsize=(4,3))
plt_tumor_data(x_train, y_train, ax)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

在下面的图里，试试：
- 在右上角的等高线图内点击来改变 $w$ 和 $b$
    - 改变可能需要一两秒
    - 注意左上图代价值的变化
    - 注意代价是由每个样本的 loss（竖直虚线）累加而成
- 点击橙色按钮运行梯度下降
    - 注意代价稳步下降（等高线图和代价图用的是 log(cost)）
    - 在等高线图里点击会重置模型、开始新的一次运行
- 要重置图，重新运行该 cell

In [20]:
w_range = np.array([-1, 7])
b_range = np.array([1, -14])
quad = plt_quad_logistic( x_train, y_train, w_range, b_range )

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 恭喜！
你已经：
- 考察了逻辑回归梯度计算的公式与实现
- 在以下两处使用了这些函数
    - 探索一个单变量数据集
    - 探索一个双变量数据集